# Georgian Spellchecker — Inference

This notebook loads the trained model and demonstrates it correcting Georgian words. It is fully self-contained, all architecture code is redefined here and the only dependency is the .pt checkpoint file produced by data_and_training.ipynb.

 Run Cell 1, then Cell 2 will open a file picker, upload georgian_spellchecker.pt file from the zip. Then run the remaining cells in order or just pres run all upload and wait for outputs.

## Cell 1 — Imports

PyTorch and the RNN utilities needed to reconstruct the model. No training libraries are required.

In [1]:
import random
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from google.colab import files as colab_files

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## Cell 2 — Upload model checkpoint

A file picker will open. Upload georgian_spellchecker.pt  

In [2]:
uploaded = colab_files.upload()

import os
pt_files = [f for f in uploaded if f.endswith('.pt')]
if not pt_files:
    raise FileNotFoundError('No .pt file found. Please upload georgian_spellchecker.pt')

MODEL_PATH = pt_files[0]
print(f'Model file: {MODEL_PATH}  ({os.path.getsize(MODEL_PATH):,} bytes)')

Saving georgian_spellchecker.pt to georgian_spellchecker.pt
Model file: georgian_spellchecker.pt  (48,850,416 bytes)


## Cell 3 — Model architecture

The same classes from training are redefined here so that PyTorch can restore the weights from the checkpoint. Nothing differs from the training notebook.

In [3]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_out, mask):
        T      = enc_out.shape[1]
        dec_h  = dec_hidden.unsqueeze(1).repeat(1, T, 1)
        energy = torch.tanh(self.attn(torch.cat([dec_h, enc_out], dim=-1)))
        scores = self.v(energy).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e9)
        return torch.softmax(scores, dim=-1)

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru       = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True,
                                bidirectional=True,
                                dropout=dropout if n_layers > 1 else 0.0)
        self.fc        = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src, src_lens=None):
        emb      = self.dropout(self.embedding(src))
        src_lens = (src != PAD).sum(dim=1).cpu().clamp(min=1)
        packed   = pack_padded_sequence(emb, src_lens, batch_first=True,
                                        enforce_sorted=False)
        out, hidden = self.gru(packed)
        out, _      = pad_packed_sequence(out, batch_first=True)
        n      = hidden.shape[0] // 2
        hidden = hidden.view(n, 2, hidden.shape[1], -1)
        hidden = torch.tanh(self.fc(torch.cat([hidden[:, 0], hidden[:, 1]], dim=-1)))
        return out, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.attention = Attention(hidden_dim)
        self.gru       = nn.GRU(embed_dim + hidden_dim * 2, hidden_dim, n_layers,
                                batch_first=True,
                                dropout=dropout if n_layers > 1 else 0.0)
        self.fc_out    = nn.Linear(hidden_dim + hidden_dim * 2, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, token, hidden, enc_out, mask=None):
        emb     = self.dropout(self.embedding(token.unsqueeze(1)))
        mask    = (enc_out.sum(dim=-1) != 0)
        attn_w  = self.attention(hidden[-1], enc_out, mask)
        context = torch.bmm(attn_w.unsqueeze(1), enc_out)
        gru_in  = torch.cat([emb, context], dim=-1)
        out, hidden = self.gru(gru_in, hidden)
        logits  = self.fc_out(torch.cat([out.squeeze(1), context.squeeze(1)], dim=-1))
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def create_mask(self, src): return (src != PAD)

print('Architecture classes defined.')

Architecture classes defined.


## Cell 4 — Load checkpoint

The checkpoint stores the model weights alongside all the hyperparameters and the character vocabulary that was used during training. Loading everything from one file means inference has no dependency on the training session.

In [4]:
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)

char2idx   = ckpt['char2idx']
idx2char   = ckpt['idx2char']
VOCAB_SIZE = ckpt['vocab_size']
EMBED_DIM  = ckpt['embed_dim']
HIDDEN_DIM = ckpt['hidden_dim']
N_LAYERS   = ckpt['n_layers']
DROPOUT    = ckpt['dropout']

enc   = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
dec   = Decoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')
print(f'Val loss at checkpoint:  {ckpt["val_loss"]:.4f}')
if 'val_acc' in ckpt:
    print(f'Val word accuracy:       {ckpt["val_acc"]*100:.1f}%')

Loaded checkpoint from epoch 60
Val loss at checkpoint:  1.1375
Val word accuracy:       42.4%


## Cell 5 — Decoding functions and correct_word

greedy_decode runs the encoder once then steps through the decoder, picking the highest-probability character at each step until EOS or max length.

correct_word is the assignment-specified function. It accepts a word and the path to the model file, loads the checkpoint internally, and returns the corrected word. It is fully self-contained and does not rely on any global session state.

In [5]:
def greedy_decode(word, max_len=40):
    model.eval()
    with torch.no_grad():
        ids     = [char2idx.get(ch, UNK) for ch in word]
        src     = torch.tensor([ids], dtype=torch.long).to(DEVICE)
        src_len = torch.tensor([len(ids)])
        enc_out, hidden = model.encoder(src, src_len)
        mask = (src != PAD)
        token   = torch.tensor([SOS], dtype=torch.long).to(DEVICE)
        result  = []
        for _ in range(max_len):
            logits, hidden = model.decoder(token, hidden, enc_out, mask)
            pred = logits.argmax(-1).item()
            if pred == EOS:
                break
            ch = idx2char.get(pred, '')
            if ch not in ('<PAD>', '<SOS>', '<EOS>', '<UNK>'):
                result.append(ch)
            token = torch.tensor([pred], dtype=torch.long).to(DEVICE)
    return ''.join(result)


def correct_word(word: str, model_path: str) -> str:
    c = torch.load(model_path, map_location=DEVICE)
    c2i, i2c = c['char2idx'], c['idx2char']

    enc_l = Encoder(c['vocab_size'], c['embed_dim'], c['hidden_dim'], c['n_layers'], c['dropout'])
    dec_l = Decoder(c['vocab_size'], c['embed_dim'], c['hidden_dim'], c['n_layers'], c['dropout'])
    m = Seq2Seq(enc_l, dec_l, DEVICE).to(DEVICE)
    m.load_state_dict(c['model_state'])
    m.eval()

    with torch.no_grad():
        ids     = [c2i.get(ch, UNK) for ch in word]
        src     = torch.tensor([ids], dtype=torch.long).to(DEVICE)
        enc_out, hidden = m.encoder(src)
        mask = (src != PAD)
        token   = torch.tensor([SOS], dtype=torch.long).to(DEVICE)
        result  = []
        for _ in range(40):
            logits, hidden = m.decoder(token, hidden, enc_out, mask)
            pred = logits.argmax(-1).item()
            if pred == EOS:
                break
            ch = i2c.get(pred, '')
            if ch not in ('<PAD>', '<SOS>', '<EOS>', '<UNK>'):
                result.append(ch)
            token = torch.tensor([pred], dtype=torch.long).to(DEVICE)
    return ''.join(result)

print('correct_word() ready.')

correct_word() ready.


## Cell 6 — Identity test

Tests whether the model leaves already-correct words unchanged. A spellchecker must do this reliably — changing correct input is just as bad as missing a real error.

In [6]:
correct_inputs = [
    'საქართველო', 'თბილისი',    'კონსტიტუცია',
    'პარლამენტი', 'განათლება',   'უნივერსიტეტი',
    'ლიტერატურა', 'სახელმწიფო', 'დამოუკიდებლობა',
    'მეცნიერება',
]

print('Identity test: correct words should pass through unchanged')
print(f'  {"Input":<25}  {"Output":<25}  Match?')
print('  ' + '-'*60)

identity_ok = 0
for w in correct_inputs:
    out   = greedy_decode(w)
    match = 'yes' if out == w else 'NO'
    if out == w:
        identity_ok += 1
    print(f'  {w:<25}  {out:<25}  {match}')

print(f'\nIdentity accuracy: {identity_ok}/{len(correct_inputs)}')

Identity test: correct words should pass through unchanged
  Input                      Output                     Match?
  ------------------------------------------------------------
  საქართველო                 საქართველო                 yes
  თბილისი                    თბილისი                    yes
  კონსტიტუცია                კონსტიტუცია                yes
  პარლამენტი                 პარლამენტი                 yes
  განათლება                  განათლება                  yes
  უნივერსიტეტი               უნივერსიტეტი               yes
  ლიტერატურა                 ლიტერატურა                 yes
  სახელმწიფო                 სახელმწიფო                 yes
  დამოუკიდებლობა             დამოუკიდებლობა             yes
  მეცნიერება                 მეცნიერება                 yes

Identity accuracy: 10/10


## Cell 7 — Correction test

Tests correction of deliberately introduced errors. Each pair is a hand-written realistic typo alongside the expected correction. The model should recover the original word in each case.

In [7]:
error_pairs = [
    ('საქრთველო',        'საქართველო'),
    ('თბილსი',           'თბილისი'),
    ('კონსტიტცია',       'კონსტიტუცია'),
    ('სახელმწფო',        'სახელმწიფო'),
    ('დამოუუიდებლობა',   'დამოუკიდებლობა'),
    ('ისტოია',           'ისტორია'),
    ('გეოგრაია',         'გეოგრაფია'),
    ('მეცნირება',        'მეცნიერება'),
    ('სამეგელო',         'სამეგრელო'),
    ('ქარტლი',           'ქართლი'),
]

print('Correction test: corrupted words should be restored')
print(f'  {"Input":<28}  {"Output":<25}  {"Target":<25}  Match?')
print('  ' + '-'*85)

correction_ok = 0
for corrupted, target in error_pairs:
    out   = greedy_decode(corrupted)
    match = 'yes' if out == target else 'NO'
    if out == target:
        correction_ok += 1
    print(f'  {corrupted:<28}  {out:<25}  {target:<25}  {match}')

print(f'\nCorrection accuracy: {correction_ok}/{len(error_pairs)}')
print(f'Total: {identity_ok + correction_ok}/{len(correct_inputs) + len(error_pairs)}')

Correction test: corrupted words should be restored
  Input                         Output                     Target                     Match?
  -------------------------------------------------------------------------------------
  საქრთველო                     საქართველო                 საქართველო                 yes
  თბილსი                        თბილისი                    თბილისი                    yes
  კონსტიტცია                    კონსტიტცია                 კონსტიტუცია                NO
  სახელმწფო                     სახელმწოფო                 სახელმწიფო                 NO
  დამოუუიდებლობა                დამოუიიდებლობა             დამოუკიდებლობა             NO
  ისტოია                        ისტორია                    ისტორია                    yes
  გეოგრაია                      გეოგრაფია                  გეოგრაფია                  yes
  მეცნირება                     მეცნირება                  მეცნიერება                 NO
  სამეგელო                      სამეგელო           

## Cell 8 — correct_word with model_path

Demonstrates the assignment-specified function signature. The model is loaded from disk inside the function, so this call is completely self-contained.

In [8]:
# the following 25 demo examples(15 corrupted, 10 clean) represent every group of corruptions represented in training
#character substitution with keyboard neighbors, transpositions with keyboard neighbours, substitutions between same key pairs, deletions and so on.
examples = [
    # substitutions and same-key confusions
    ('შყალი',  'წყალი'),
    ('ქარტული',  'ქართული'),
    ('სიყვაღული', 'სიყვარული'),
    ('ტბილისი', 'თბილისი'),
    ('გამარჰობა', 'გამარჯობა'),
    ('საპრეზიდენთო', 'საპრეზიდენტო'),
    # transpositions
    ('ინსრტუმენტი', 'ინსტრუმენტი'),
    ('პარლამნეტი', 'პარლამენტი'),
    # deletions
    ('გეოგრაია', 'გეოგრაფია'),
    ('ღმრთი', 'ღმერთი'),
    ('პროგამა', 'პროგრამა'),
    ('ხელოვნბა', 'ხელოვნება'),
    ('განსაკუთრებუი', 'განსაკუთრებული'),
    #Double character
    ('ბააასი', 'ბაასი'),
    ('ქაალაქი','ქალაქი'),
    # clean passthroughs
    ('ვეფხისტყაოსანი', 'ვეფხისტყაოსანი'),
    ('თბილისი', 'თბილისი'),
    ('კულტურა', 'კულტურა'),
    ('განათლება','განათლება'),
    ('ეკონომიკა','ეკონომიკა'),
    ('მუსიკა', 'მუსიკა'),
    ('სპორტი', 'სპორტი'),
    ('მედიცინა','მედიცინა'),
    ('ისტორია', 'ისტორია'),
    ('ფიზიკა','ფიზიკა'),
]

print('correct_word(word, model_path) — 20 example demonstration:')
print(f'  {"Input":<25}  {"Output":<25}  {"Expected":<25}  Match?')
print('  ' + '-'*80)
total_ok = 0
for corrupted, expected in examples:
    out   = correct_word(corrupted, MODEL_PATH)
    match = 'yes' if out == expected else 'NO'
    if out == expected:
        total_ok += 1
    print(f'  {corrupted:<25}  {out:<25}  {expected:<25}  {match}')

print(f'\nTotal: {total_ok}/{len(examples)}')


correct_word(word, model_path) — 20 example demonstration:
  Input                      Output                     Expected                   Match?
  --------------------------------------------------------------------------------
  შყალი                      წყალი                      წყალი                      yes
  ქარტული                    ქართული                    ქართული                    yes
  სიყვაღული                  სიყვარული                  სიყვარული                  yes
  ტბილისი                    თბილისი                    თბილისი                    yes
  გამარჰობა                  გამარჯობა                  გამარჯობა                  yes
  საპრეზიდენთო               საპრეზიდენტო               საპრეზიდენტო               yes
  ინსრტუმენტი                ინსტუმენტი                 ინსტრუმენტი                NO
  პარლამნეტი                 პარლამენტი                 პარლამენტი                 yes
  გეოგრაია                   გეოგრაფია                  გეოგრაფია        